# Volume 8, Chapter 3: Differential Forms and Stokes (Python companion)

Stokes' theorem at work in computational geometry: areas from boundaries, winding numbers for inside-outside tests, and their robust generalization to broken polygons and meshes.

*Before each solution, predict what the output will look like.*

## Exercises

**P3.1 (★) CS application: areas of map polygons.** (a) Generate a polygon with a few thousand vertices that looks like a coastline (a star-shaped polygon with a fractal-looking random radius), and compute its area with the shoelace formula of Exercise 3.2, with Monte Carlo sampling, and by counting pixels on a grid. Compare accuracy and cost. (b) For a self-intersecting polygon (a pentagram, or a random closed polyline), show that the shoelace sum equals $\sum_k k\cdot\operatorname{area}\{p:n(\gamma,p)=k\}$, where $n$ is the winding number: compute the winding number on a grid with the integral $\frac1{2\pi}\int_\gamma\dd\theta$ (Corollary 3.21).

**P3.2 (★★) CS application: inside or outside? Winding numbers versus ray casting.** (a) Implement the point-in-polygon test in two ways: by ray casting (count crossings of a horizontal ray, parity) and by the winding number $\frac1{2\pi}\sum_{\text{edges}}\angle(p_i-p,\,p_{i+1}-p)$. Check that they agree on simple polygons. (b) Now damage the polygon: delete a few edges so that it is no longer closed. Show that ray casting produces long streaks of wrong answers, while the *generalized winding number* (the same angle sum, now a real number) degrades gracefully: thresholding it at $\frac12$ still classifies almost every point correctly. Explain this with Stokes' theorem.

**P3.3 (★★) The generalized winding number of a triangle mesh.** In $\R^3$ the role of $\dd\theta$ is played by the closed $2$-form $\frac{\sigma}{|x|^3}$ on $\R^3\setminus\{0\}$, whose integral over a closed surface is $4\pi$ times the winding number. For a triangle mesh this integral is a sum of solid angles of triangles, given by the formula of Van Oosterom and Strackee. (a) Implement it and check on an octahedron and on a subdivided sphere that it gives $1$ inside and $0$ outside. (b) Remove a few triangles (a hole in the mesh) and compare with ray-parity tests. (c) Check that the solid-angle sum varies continuously across the hole and jumps by $1$ across the surface.

## Solutions

### P3.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
m = 4000
t = np.linspace(0, 2 * np.pi, m, endpoint=False)
# a random radius with power-law Fourier coefficients: a rough "coastline"
k = np.arange(1, 400)
r = 1.0 + 0.25 * np.sum((rng.standard_normal((len(k), 1)) * np.cos(np.outer(k, t) + rng.uniform(0, 2*np.pi, (len(k), 1))))
                        / k[:, None] ** 1.1, axis=0)
x, y = r * np.cos(t), r * np.sin(t)

def shoelace(x, y):
    return 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)

def winding(px, py, x, y):
    """generalized winding number of the closed polyline (x, y) around each point (px, py)"""
    ax, ay = x[None, :] - px[:, None], y[None, :] - py[:, None]
    bx, by = np.roll(ax, -1, axis=1), np.roll(ay, -1, axis=1)
    ang = np.arctan2(ax * by - ay * bx, ax * bx + ay * by)     # signed angle from a to b
    return ang.sum(axis=1) / (2 * np.pi)

A = shoelace(x, y)
print(f"shoelace area: {A:.6f}")

# exact polar formula for a star-shaped polygon is the same sum of triangles; compare with sampling
R = np.abs(np.r_[x, y]).max() * 1.01
for n in (10_000, 100_000):
    P = rng.uniform(-R, R, (n, 2))
    inside = np.zeros(n, bool)
    for s in range(0, n, 2000):
        inside[s:s+2000] = np.abs(winding(P[s:s+2000, 0], P[s:s+2000, 1], x, y)) > 0.5
    est = inside.mean() * (2 * R) ** 2
    print(f"Monte Carlo with {n:7d} points: {est:.4f}  (error {est - A:+.4f}, std ~ {(2*R)**2*np.sqrt(inside.mean()*(1-inside.mean())/n):.4f})")

from matplotlib.path import Path
poly = Path(np.c_[x, y])
for g in (100, 400, 1600):
    gx = (np.arange(g) + 0.5) / g * 2 * R - R
    X, Y = np.meshgrid(gx, gx)
    cnt = poly.contains_points(np.c_[X.ravel(), Y.ravel()]).sum()
    est = cnt * (2 * R / g) ** 2
    print(f"pixel count on a {g}x{g} grid: {est:.5f}  (error {est - A:+.5f})")
plt.figure(figsize=(4.5, 4.5)); plt.fill(x, y, alpha=0.4); plt.plot(x, y, lw=0.4); plt.axis("equal"); plt.title("a random coastline"); plt.show()

In [ ]:
# (b) a self-intersecting polygon: the pentagram
ang = np.pi / 2 + 4 * np.pi / 5 * np.arange(5)
x5, y5 = np.cos(ang), np.sin(ang)
S = shoelace(x5, y5)
g = 800
gx = (np.arange(g) + 0.5) / g * 2.2 - 1.1
X, Y = np.meshgrid(gx, gx)
W = np.rint(winding(X.ravel(), Y.ravel(), x5, y5)).astype(int)
cell = (2.2 / g) ** 2
for k in sorted(set(W)):
    print(f"winding number {k}: area {np.sum(W == k) * cell:.4f}")
print(f"sum of k * area = {np.sum(W) * cell:.4f},  shoelace = {S:.4f}")

(a) The shoelace formula is exact for the polygon (up to rounding) and costs one pass over the $m$ vertices. Monte Carlo with $n$ points has a statistical error of order $n^{-1/2}$ (standard deviation about $0.02$ for $10^4$ points and $0.008$ for $10^5$), and each point costs a full inside test. Pixel counting with $g\times g$ pixels is only wrong on the pixels that meet the boundary, so its error is at most the perimeter times the pixel size, $O(\frac1g)$; here the errors of opposite sign along the boundary largely cancel and it does better, but it needs $g^2$ tests. Stokes' theorem turns a two-dimensional integral into a one-dimensional one, and for a polygon the one-dimensional integral is a finite sum: that is why geographic software computes areas from boundaries.

(b) For the pentagram the central pentagon has winding number $2$ and the five points have winding number $1$. The shoelace sum equals $\sum_kk\cdot\operatorname{area}_k$ (to the grid accuracy), not the area of the star: for a closed curve $\gamma$, $\frac12\int_\gamma(x\dd y-y\dd x)=\int_{\R^2}n(\gamma,p)\dd p$, which is Stokes applied to each region where the winding number is constant. Only for a simple anticlockwise polygon is $n$ the indicator of the inside.

### P3.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def ray_parity(px, py, x, y, closed=True):
    """count crossings of the ray to the right of each point; segments i -> i+1"""
    n = len(x)
    idx = range(n if closed else n - 1)
    c = np.zeros(len(px), int)
    for i in idx:
        x1, y1, x2, y2 = x[i], y[i], x[(i + 1) % n], y[(i + 1) % n]
        cond = (y1 > py) != (y2 > py)
        xc = x1 + (py - y1) * (x2 - x1) / (y2 - y1 + 1e-300)
        c += cond & (px < xc)
    return c % 2 == 1

def gwn(px, py, x, y, closed=True):
    ax, ay = x[None, :] - px[:, None], y[None, :] - py[:, None]
    bx, by = np.roll(ax, -1, axis=1), np.roll(ay, -1, axis=1)
    a = np.arctan2(ax * by - ay * bx, ax * bx + ay * by)
    if not closed:
        a = a[:, :-1]
    return a.sum(axis=1) / (2 * np.pi)

# a non-convex polygon: a "C" shape
t = np.linspace(0.35, 2 * np.pi - 0.35, 60)
outer = np.c_[np.cos(t), np.sin(t)]
inner = np.c_[0.55 * np.cos(t[::-1]), 0.55 * np.sin(t[::-1])]
P = np.vstack([outer, inner])
x, y = P[:, 0], P[:, 1]

g = 300
gx = np.linspace(-1.2, 1.2, g)
X, Y = np.meshgrid(gx, gx); px, py = X.ravel(), Y.ravel()
truth = ray_parity(px, py, x, y)
w = gwn(px, py, x, y)
print("closed polygon: ray parity and winding number disagree at", np.sum(truth != (np.abs(w) > 0.5)), "of", g * g, "points")

# (b) damage: delete 6 consecutive edges of the outer arc (keep the vertices, drop the segments)
# the polyline now runs from vertex 26 round to vertex 20 and is not closed
order = np.r_[np.arange(26, len(x)), np.arange(0, 21)]
xd, yd = x[order], y[order]
ray_d = ray_parity(px, py, xd, yd, closed=False)
w_d = gwn(px, py, xd, yd, closed=False)
print("damaged polygon: ray parity wrong at", np.sum(ray_d != truth), "points;",
      "generalized winding number (threshold 1/2) wrong at", np.sum((w_d > 0.5) != truth), "points")

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(truth.reshape(g, g), origin="lower", extent=[-1.2, 1.2, -1.2, 1.2]); ax[0].set_title("true inside")
ax[1].imshow(ray_d.reshape(g, g), origin="lower", extent=[-1.2, 1.2, -1.2, 1.2]); ax[1].set_title("ray parity, damaged")
im = ax[2].imshow(w_d.reshape(g, g), origin="lower", extent=[-1.2, 1.2, -1.2, 1.2], cmap="viridis")
ax[2].set_title("generalized winding number, damaged"); plt.colorbar(im, ax=ax[2], fraction=0.046)
for a in ax[1:]: a.plot(xd, yd, 'r-', lw=0.8)
plt.tight_layout(); plt.show()

(a) On a closed polygon the two tests agree everywhere (except possibly at points exactly on the boundary). The winding number is the integral of the closed form $\dd\theta_p$ along the boundary, and the ray test counts the signed crossings of a ray: two ways of computing the same integer, since $\dd\theta_p$ is locally the differential of the angle and its integral only changes when the curve crosses the cut along the ray.

(b) With six edges deleted, ray casting is wrong at about $2800$ of the $90000$ grid points, on every horizontal line through the gap: once a crossing is missed the parity flips for every point on that line, producing streaks across the whole shape. The generalized winding number is a real number, the angle under which the remaining curve is seen from $p$ divided by $2\pi$. Deleting a small arc changes it only by the angle that arc subtends, which is small except near the gap itself, so thresholding at $\frac12$ is wrong only in a small region near the gap (about $220$ points here). In terms of Stokes' theorem: the winding number of the closed curve is the integral of $\dd\theta_p$, and removing an arc subtracts the integral over that arc, a continuous function of $p$ away from the arc. This idea, applied to triangle meshes in $\R^3$ (next problem), is the robust inside-outside test of Jacobson, Kavan and Sorkine-Hornung (2013), now standard in geometry processing.

### P3.3

In [ ]:
import numpy as np

def solid_angles(p, V, F):
    """sum over triangles of the signed solid angle (Van Oosterom-Strackee 1983) seen from each point of p"""
    total = np.zeros(len(p))
    for f in F:
        a, b, c = V[f[0]] - p, V[f[1]] - p, V[f[2]] - p
        la, lb, lc = np.linalg.norm(a, axis=1), np.linalg.norm(b, axis=1), np.linalg.norm(c, axis=1)
        num = np.einsum('ij,ij->i', a, np.cross(b, c))
        den = la * lb * lc + np.einsum('ij,ij->i', a, b) * lc + np.einsum('ij,ij->i', b, c) * la + np.einsum('ij,ij->i', c, a) * lb
        total += 2 * np.arctan2(num, den)
    return total / (4 * np.pi)

# octahedron, faces oriented outwards
V = np.array([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1]], float)
F = []
for sx, ix in ((1, 0), (-1, 1)):
    for sy, iy in ((1, 2), (-1, 3)):
        for sz, iz in ((1, 4), (-1, 5)):
            tri = [ix, iy, iz]
            if sx * sy * sz < 0:
                tri = [ix, iz, iy]
            F.append(tri)
F = np.array(F)

pts = np.array([[0, 0, 0], [0.2, 0.1, -0.3], [0.5, 0.5, 0.5], [2, 0, 0], [0, -3, 1]])
print("octahedron:", np.round(solid_angles(pts, V, F), 6))

def icosphere(levels):
    t = (1 + 5 ** 0.5) / 2
    V = [[-1, t, 0], [1, t, 0], [-1, -t, 0], [1, -t, 0], [0, -1, t], [0, 1, t], [0, -1, -t], [0, 1, -t],
         [t, 0, -1], [t, 0, 1], [-t, 0, -1], [-t, 0, 1]]
    F = [[0, 11, 5], [0, 5, 1], [0, 1, 7], [0, 7, 10], [0, 10, 11], [1, 5, 9], [5, 11, 4], [11, 10, 2], [10, 7, 6],
         [7, 1, 8], [3, 9, 4], [3, 4, 2], [3, 2, 6], [3, 6, 8], [3, 8, 9], [4, 9, 5], [2, 4, 11], [6, 2, 10], [8, 6, 7], [9, 8, 1]]
    V = [np.array(v) / np.linalg.norm(v) for v in V]
    for _ in range(levels):
        cache = {}
        def mid(i, j):
            key = (min(i, j), max(i, j))
            if key not in cache:
                v = V[i] + V[j]; V.append(v / np.linalg.norm(v)); cache[key] = len(V) - 1
            return cache[key]
        F = [[a, mid(a, b), mid(c, a)] for a, b, c in F] + [[b, mid(b, c), mid(a, b)] for a, b, c in F] + \
            [[c, mid(c, a), mid(b, c)] for a, b, c in F] + [[mid(a, b), mid(b, c), mid(c, a)] for a, b, c in F]
    return np.array(V), np.array(F)

V2, F2 = icosphere(2)
print("icosphere:", len(V2), "vertices,", len(F2), "triangles")
print("  winding at centre and outside:", np.round(solid_angles(np.array([[0, 0, 0.], [0, 0, 2.]]), V2, F2), 6))

In [ ]:
# (b) and (c): remove 6 triangles around the north pole and probe along the z-axis
north = np.argmax(V2[:, 2])
holes = [i for i, f in enumerate(F2) if north in f]
Fh = np.delete(F2, holes, axis=0)
print("removed", len(holes), "triangles")

d = V2[north]                                    # probe along the line through the centre of the hole
perp = np.cross(d, [0.3, 0.5, 0.8]); perp /= np.linalg.norm(perp)
zs = np.linspace(-1.45, 1.45, 12)
probe = zs[:, None] * d[None, :] + 0.01 * perp[None, :]
w_full, w_hole = solid_angles(probe, V2, F2), solid_angles(probe, V2, Fh)

def ray_parity_mesh(p, V, F, d=np.array([0.0, 0.0, 1.0])):
    """Moeller-Trumbore crossings of the ray p + s d, s > 0"""
    count = np.zeros(len(p), int)
    for f in F:
        v0, v1, v2 = V[f]
        e1, e2 = v1 - v0, v2 - v0
        h = np.cross(d, e2); a = e1 @ h
        if abs(a) < 1e-14: continue
        s = p - v0; u = (s @ h) / a
        q = np.cross(s, e1); v = (q @ d) / a; t = (q @ e2) / a
        count += (u >= 0) & (v >= 0) & (u + v <= 1) & (t > 1e-12)
    return count % 2 == 1

par = ray_parity_mesh(probe, V2, Fh, d)
par_full = ray_parity_mesh(probe, V2, F2, d)
print("   s     full mesh   holed mesh   parity full   parity holed  (ray towards the hole)")
for z, a, b, e, c in zip(zs, w_full, w_hole, par_full, par):
    print(f"{z:5.2f}   {a:9.4f}   {b:10.4f}   {str(e):>10s}   {str(c):>10s}")

(a) The solid-angle sum is $1$ at interior points and $0$ outside, to rounding error, for both meshes: it is the integral of the closed form $\frac{\sigma}{|x-p|^3}$ over the surface, which by Stokes (applied between the surface and a small sphere around $p$) is $4\pi$ if $p$ is inside and $0$ if outside.

(b)--(c) With a hole cut around one vertex, rays shot towards the hole from points inside the sphere pass through it and report ``outside'' for every one of them, while rays from outside on the far side cross the surface once and report ``inside'': the parity test is wrong on the whole probe line. The intact mesh gives the right parity everywhere. The generalized winding number is still close to $1$ deep inside (the missing triangles subtend a small solid angle from there), decreases continuously as $p$ approaches the hole, passes $\frac12$ near the hole itself, and is close to $0$ outside. Across an intact part of the surface (the south pole) it jumps by exactly $1$. This is the three-dimensional version of P3.2, and it is the reason the generalized winding number is used to clean up the imperfect meshes that come out of scanners: it is a harmonic function of $p$ away from the surface, so it degrades gracefully where the data are missing.